# MiniGPT-Stack SFT launcher (Phase 4a) — THIN, all logic lives in train/

One Save Version = one MODE. Set MODE, run top to bottom, Save Version.

- `probe`  : **step 1 of the run.** Measures seconds/update and tok/s at
  `--batch_seqs` 8, 32 and 64, with a startup intercept wide enough to
  resolve. Writes `/kaggle/working/sft_probe.json`. No real training.
- `sweep`  : LR sweep, a fresh ckpt_dir per arm (in `/tmp`, so arm checkpoints
  and their `metrics_sft.jsonl` never collide and never enter the output).
- `final`  : the real run at the winning LR. Persistent and resumable.

Shared cells (GPU check, clone, install+tests, shards) are copied from
`kaggle_pretrain.ipynb` so the two launchers cannot drift.

**Attach two datasets:** `tinystories-shards-8192` and
`glassbox-sft-init-weights` (weights_step12158.pt, exactly 125,893,589 bytes).


## Pinned flags — change only with a decisions.md note

`MAX_EXAMPLES`/`EPOCHS`/`BATCH_SEQS`/`SEED` define `steps_per_epoch`, so they
**must be identical** across every resume of the same run. Changing any of them
silently reinterprets the step stored in an existing checkpoint.


In [ ]:
MODE = "probe"   # probe | sweep | final   <-- SET THIS, then run all

# ---- PINNED (Phase 4 plan) ----
MAX_EXAMPLES   = 20_000   # cap on train examples; 0 = whole split (~2M, OOMs)
EPOCHS         = 2
BATCH_SEQS     = 8
SEED           = 0
SAVE_EVERY_MIN = 10       # periodic checkpoint backstop
BASE_LR        = 1e-4
LR_GRID        = [5e-5, 1e-4, 2e-4]
MAX_HOURS      = 3.0      # clean save+exit before Kaggle kills the session
WEIGHTS_FILE   = "weights_step12158.pt"
WEIGHTS_BYTES  = 125_893_589
BEST_LR        = None     # <-- set from the sweep summary, then MODE="final"

assert MODE in ("probe", "sweep", "final"), MODE


def log_run(tag, ckpt_dir, lr, batch=None):
    """Print the arm and every pinned flag as the FIRST lines of its log."""
    b = BATCH_SEQS if batch is None else batch
    print("=" * 78)
    print(f"[sft] RUN arm={tag} lr={lr:g} ckpt_dir={ckpt_dir}")
    print(f"[sft] flags max_examples={MAX_EXAMPLES} batch_seqs={b} "
          f"epochs={EPOCHS} seed={SEED} save_every_min={SAVE_EVERY_MIN:g} "
          f"max_hours={MAX_HOURS:g}")
    print("=" * 78, flush=True)

## GPU check (copied from kaggle_pretrain.ipynb)

In [ ]:
import torch
print("gpu", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (prep mode)")
print("torch", torch.__version__, "| cuda build", torch.version.cuda,
      "| cuda avail", torch.cuda.is_available())
assert torch.cuda.is_available(), "enable GPU in notebook settings"


## Clone (copied from kaggle_pretrain.ipynb)

In [ ]:
import os
REPO = "https://github.com/Priyanshaur/Glassbox.git"
ROOT = "/kaggle/working/Glassbox"
if not os.path.exists(ROOT):
    !git clone $REPO $ROOT
else:
    !git -C $ROOT pull --ff-only
%cd $ROOT
!git rev-parse --short HEAD

## Install + shard-independent tests (copied from kaggle_pretrain.ipynb)

Only the three suites that need no data shards run here. `test_phase4.py`
needs `data/meta.json`, so it runs in its own cell after the shards are linked.


In [ ]:
# Kaggle already ships numpy, pyyaml, tqdm, pytest, regex and datasets. Do NOT install the
# pinned requirements.txt here: it downgrades numpy and rebuilds it from source (~3 min).
!pip install -q regex pyyaml pytest tqdm datasets
!python -m pytest tests/test_config.py tests/test_model.py tests/test_pretrain.py -q 2>&1 | tail -n 2

## Inputs — init weights (private dataset) and TinyStories shards

The weights size is asserted exactly, then loaded into the configured arch.
A truncated upload or a mismatched checkpoint fails here, not 20 minutes in.


In [ ]:
import glob, os, torch

hits = glob.glob("/kaggle/input/**/" + WEIGHTS_FILE, recursive=True)
assert hits, f"attach the private dataset containing {WEIGHTS_FILE}"
W = hits[0]
got = os.path.getsize(W)
print("weights:", W)
print("bytes  :", got, "(expected", WEIGHTS_BYTES, ")")
assert got == WEIGHTS_BYTES, f"weights size {got} != {WEIGHTS_BYTES} (truncated upload?)"

from common.config import load_config
from model.gpt import GPT, count_params

cfg = load_config("configs/base_30m.yaml")
_probe_model = GPT(cfg["model"])
_probe_model.load_state_dict(torch.load(W, map_location="cpu", weights_only=True))
print("loads into the configured arch:", True)
print("params:", count_params(_probe_model))
del _probe_model

## Shards (copied from kaggle_pretrain.ipynb; prep branch removed)

In [ ]:
import glob, json, os
hits = glob.glob("/kaggle/input/**/meta.json", recursive=True)
assert hits, "attach the shards dataset first"
print("meta.json candidates:", hits)
shard_dir = os.path.dirname(hits[0])
print("--- meta.json head ---")
print("\n".join(open(os.path.join(shard_dir, "meta.json")).read().splitlines()[:15]))
os.makedirs("data", exist_ok=True)
# The cloned repo can contain a stale data/meta.json (smoke-test counts). If it is left in
# place, the symlink below is skipped and training would read the wrong token counts.
for f in glob.glob("data/*.bin") + ["data/meta.json"]:
    if os.path.lexists(f):
        os.remove(f)
for f in glob.glob(os.path.join(shard_dir, "*.bin")) + [os.path.join(shard_dir, "meta.json")]:
    os.symlink(f, os.path.join("data", os.path.basename(f)))
print("linked:", sorted(os.listdir("data")))
meta = json.load(open("data/meta.json"))
assert "shards" in meta and "vocab_size" in meta, meta.keys()
for s in meta["shards"]:
    assert {"split", "index", "file", "tokens"} <= set(s), s
print("meta keys OK:", sorted(meta.keys()))
tr = sum(s["tokens"] for s in meta["shards"] if s["split"] == "train")
va = sum(s["tokens"] for s in meta["shards"] if s["split"] == "validation")
print(f"EXACT counts from data/meta.json: train={tr} val={va} vocab={meta['vocab_size']}")
for s in meta["shards"]:
    nbytes = os.path.getsize(os.path.join("data", s["file"]))
    print(f"  {s['file']} {s['tokens']} toks {nbytes/1024/1024:.1f}MB")
    assert nbytes == 2 * s["tokens"], f"{s['file']}: file size does not match meta (stale meta.json?)"
assert tr > 400_000_000, f"train tokens {tr} look wrong (expected ~467M)"
TRAIN_TOKS, VAL_TOKS = tr, va

## Phase 4 tests — after the shards, and fatal on failure

These tests read `data/meta.json`, so they cannot run with the early suites.
They are run through `subprocess` so a non-zero exit raises: a `!pytest` magic
would only print the failure and let training continue on a broken tree.


In [ ]:
import subprocess, sys

proc = subprocess.run([sys.executable, "-m", "pytest",
                       "tests/test_phase4.py", "-q"])
if proc.returncode != 0:
    raise RuntimeError(f"[sft] tests/test_phase4.py FAILED "
                       f"(rc={proc.returncode}); stopping")
print("[sft] tests/test_phase4.py: PASS", flush=True)

## Restore previous SFT checkpoints (final only)

SFT-only guard: a Phase 3 pretrain checkpoint shares the arch, so
`load_checkpoint` would accept it as SFT progress. An SFT ckpt_dir is the only
place `metrics_sft.jsonl` exists, so require it.


In [ ]:
import glob, os, re, shutil
if MODE != "final":
    print("skip: only the final run restores previous checkpoints")
else:
    cands = [p for p in glob.glob("/kaggle/input/**/ckpt_step*.pt", recursive=True)
             if "nan" not in os.path.basename(p)
             and os.path.exists(os.path.join(os.path.dirname(p), "metrics_sft.jsonl"))]
    def _step(p):
        m = re.search(r"ckpt_step(\d+)\.pt", os.path.basename(p))
        return int(m.group(1)) if m else -1
    cands = sorted(cands, key=_step)
    dst_dir = "/kaggle/working/checkpoints/sft_final"  # ABSOLUTE: cwd is the clone dir
    if cands:
        newest, src_dir = cands[-1], os.path.dirname(cands[-1])
        print(f"newest SFT checkpoint by numeric step: {newest} (step {_step(newest)})")
        os.makedirs(dst_dir, exist_ok=True)
        for f in glob.glob(os.path.join(src_dir, "*")):
            if os.path.basename(f) not in os.listdir(dst_dir):
                shutil.copy(f, dst_dir)
        print("restored", len(os.listdir(dst_dir)), "files ->", dst_dir)
    else:
        print("no previous SFT checkpoints attached; starting fresh")

## STEP 1 OF THE RUN — timing probe at batch_seqs 8 / 32 / 64

Startup is calibrated from two batch-8 runs 125 updates apart, so the intercept
is larger than timer noise (the old 25-vs-12 gap gave a negative intercept).

Per batch: `sec/update`, `tok/update = batch_seqs * mean_example_tokens`, and
`tok/s`. The larger batches reuse the batch-8 startup, so one run each suffices.
An OOM at one batch size is caught and reported as skipped; the probe continues
with the remaining sizes (a handled OOM is not a crash and is not retried).
Probe checkpoints go to `/tmp` and are deleted after timing.


In [ ]:
import json, os, shutil, subprocess, time

PROBE = "/kaggle/working/sft_probe.json"
PROBE_BATCHES = [8, 32, 64]
BIG_EXAMPLES, SMALL_EXAMPLES = 1200, 200   # batch 8: 150 vs 25 updates


def _is_oom(rc, tail):
    """A per-batch OOM is handled, not a crash: SIGKILL or a CUDA OOM message."""
    blob = "\n".join(tail).lower()
    return rc == -9 or "out of memory" in blob or "outofmemoryerror" in blob


def probe(nex, batch, tag):
    d = f"/tmp/probe_{tag}"
    shutil.rmtree(d, ignore_errors=True)
    log_run(f"probe-{tag}", d, BASE_LR, batch=batch)
    argv = ["python", "-m", "train.sft",
            "--config", "configs/base_30m.yaml",
            "--init_weights", W, "--ckpt_dir", d,
            "--max_examples", str(nex), "--epochs", "1",
            "--batch_seqs", str(batch), "--lr", str(BASE_LR),
            "--seed", str(SEED), "--save_every", "0", "--save_every_min", "999",
            "--max_hours", "0", "--resume", "none"]
    t0 = time.time()
    # Stream the child's output into the kernel log but keep the tail, so an
    # OOM at one batch size is recognised and skipped instead of ending the run.
    p = subprocess.Popen(argv, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        print(line, end="", flush=True)
        tail.append(line.rstrip())
        if len(tail) > 60:
            tail.pop(0)
    rc = p.wait()
    dt = time.time() - t0
    updates = max(nex // batch, 1)
    if rc != 0:
        if _is_oom(rc, tail):
            print(f"[sft] OOM at batch={batch} after {dt:.1f}s; "
                  f"skipping this batch and continuing (no retry)", flush=True)
            shutil.rmtree(d, ignore_errors=True)
            return None
        raise RuntimeError(f"[sft] FAILED: probe {tag} exited with code {rc}")
    shutil.rmtree(d, ignore_errors=True)
    print(f"[sft] probe {tag}: batch={batch} nex={nex} updates={updates} "
          f"wall={dt:.2f}s", flush=True)
    return dt, updates


if MODE != "probe":
    print("skip: probe runs in probe mode only")
elif os.path.exists(PROBE):
    print("already measured this kernel:", json.load(open(PROBE)))
else:
    t_big, u_big = probe(BIG_EXAMPLES, 8, "b8-big")
    t_small, u_small = probe(SMALL_EXAMPLES, 8, "b8-small")
    if t_big is None or t_small is None:
        raise RuntimeError("[sft] batch 8 OOM: startup cannot be calibrated")
    assert u_big > u_small, (u_big, u_small)
    sec8 = (t_big - t_small) / (u_big - u_small)
    raw_startup = t_small - u_small * sec8
    startup = max(raw_startup, 0.0)
    if raw_startup < 0:
        print(f"[sft] WARNING: startup measured {raw_startup:.2f}s (noise); using 0")

    per = {8: {"updates": u_big, "wall_s": t_big, "sec_per_update": sec8}}
    oom_skipped = []
    for b in PROBE_BATCHES:
        if b == 8:
            continue
        r = probe(BIG_EXAMPLES, b, f"b{b}")
        if r is None:
            oom_skipped.append(b)
            continue
        t, u = r
        sec = (t - startup) / u
        if sec <= 0:
            print(f"[sft] WARNING: non-positive sec/update at batch={b}")
            sec = 1e-6
        per[b] = {"updates": u, "wall_s": t, "sec_per_update": sec}

    # Mean example length measured from the same streamed slice training uses.
    from common.config import load_config
    from tokenizer.bpe import BPETokenizer
    from train.sft import collect_examples
    cfg = load_config("configs/base_30m.yaml")
    tk = BPETokenizer.load(cfg["data"]["tokenizer_path"])
    ex, _, _ = collect_examples(tk, "data", "train", cfg["train"]["context_len"], 200)
    mean_len = sum(len(e["input"]) for e in ex) / len(ex)

    for b, m in per.items():
        m["tokens_per_update"] = b * mean_len
        m["tok_s"] = m["tokens_per_update"] / m["sec_per_update"]
        m["startup_s"] = startup

    best = max(per, key=lambda b: per[b]["tok_s"])
    full_updates = EPOCHS * max(MAX_EXAMPLES // BATCH_SEQS, 1)
    predicted = startup + full_updates * per[BATCH_SEQS]["sec_per_update"]
    info = {"batches": {str(b): m for b, m in per.items()},
            "batch_seqs_used": BATCH_SEQS,
            "oom_skipped": oom_skipped,
            "mean_example_tokens": mean_len,
            "startup_s": startup,
            "sec_per_update": per[BATCH_SEQS]["sec_per_update"],
            "tok_s": per[BATCH_SEQS]["tok_s"],
            "full_updates": full_updates,
            "predicted_full_run_s": predicted,
            "best_tok_s_batch": best}
    json.dump(info, open(PROBE, "w"), indent=2)

    print("\n[sft] === MEASURED (not guessed) ===")
    print(f"[sft] startup = {startup:.3f} s "
          f"(from {u_big} vs {u_small} updates at batch 8)")
    print(f"[sft] {'batch':>5} {'s/update':>9} {'tok/update':>11} {'tok/s':>8}")
    for b in sorted(per):
        m = per[b]
        print(f"[sft] {b:>5} {m['sec_per_update']:>9.4f} "
              f"{m['tokens_per_update']:>11.0f} {m['tok_s']:>8.0f}")
    for b in oom_skipped:
        print(f"[sft] {b:>5} {'OOM':>9} {'(skipped)':>11} {'-':>8}")
    print(f"[sft] fastest batch: {best} ({per[best]['tok_s']:.0f} tok/s)")
    print(f"[sft] full run at batch_seqs={BATCH_SEQS}: {full_updates} updates, "
          f"predicted {predicted/60:.1f} min")
    if predicted > MAX_HOURS * 3600:
        print(f"[sft] WARNING: exceeds --max_hours={MAX_HOURS:g}; it will exit "
              "early and resume next session")
    print("[sft] copied to /kaggle/working/sft_probe.json")

## LR sweep — one fresh ckpt_dir per arm

Arms live in `/tmp` and are wiped before each run, so the append-only
`metrics_sft.jsonl` of one arm can never be read as another's progress. Only
each arm's final `weights_step*.pt` is copied into the saved output.
Best arm = lowest **response-token val loss**.


In [ ]:
import glob, json, os, re, shutil, subprocess

TAGS = {}

if MODE != "sweep":
    print("skip: sweep runs in sweep mode only")
else:
    try:
        info = json.load(open("/kaggle/working/sft_probe.json"))
    except FileNotFoundError:
        # Every kernel version starts with a fresh /kaggle/working, so the
        # probe json never carries over between versions.
        info = None
        print("[sft] sft_probe.json absent (fresh /kaggle/working this version): "
              "the max_hours prediction check is skipped. Running with the "
              "pinned flags.")
    if info:
        per_arm_min = (info["sec_per_update"] * EPOCHS
                       * max(MAX_EXAMPLES // BATCH_SEQS, 1)) / 60
        print(f"[sft] measured {info['sec_per_update']:.4f} s/update "
              f"({info['tok_s']:.0f} tok/s)")
        print(f"[sft] ~{per_arm_min:.1f} min/arm x {len(LR_GRID)} arms "
              f"= ~{per_arm_min * len(LR_GRID):.0f} min total")

    for lr in LR_GRID:
        tag = f"lr{lr:g}".replace(".", "p").replace("-", "m")
        TAGS[lr] = tag
        d = f"/tmp/sft_{tag}"
        shutil.rmtree(d, ignore_errors=True)        # FRESH dir per arm
        log_run(tag, d, lr)
        argv = ["python", "-m", "train.sft",
                "--config", "configs/base_30m.yaml",
                "--init_weights", W, "--ckpt_dir", d,
                "--max_examples", str(MAX_EXAMPLES), "--epochs", str(EPOCHS),
                "--batch_seqs", str(BATCH_SEQS), "--lr", str(lr),
                "--seed", str(SEED), "--save_every", "0",
                "--save_every_min", str(SAVE_EVERY_MIN),
                "--max_hours", str(MAX_HOURS), "--resume", "latest"]
        proc = subprocess.run(argv)
        if proc.returncode != 0:
            raise RuntimeError(f"[sft] FAILED: arm {tag} exited with code "
                               f"{proc.returncode}")
        ws = sorted(glob.glob(os.path.join(d, "weights_step*.pt")),
                    key=lambda x: int(re.search(r"weights_step(\d+)", x).group(1)))
        os.makedirs("/kaggle/working/checkpoints", exist_ok=True)
        if ws:
            dst = f"/kaggle/working/checkpoints/sft_{tag}_weights.pt"
            shutil.copy(ws[-1], dst)
            print(f"[sft] kept {dst} ({os.path.getsize(dst)} bytes)", flush=True)

    print("\n[sft] === SWEEP SUMMARY (response-token val loss; lower is better) ===")
    best = None
    for lr in LR_GRID:
        tag = TAGS.get(lr) or f"lr{lr:g}".replace(".", "p").replace("-", "m")
        mf = f"/tmp/sft_{tag}/metrics_sft.jsonl"
        if not os.path.exists(mf):
            print(f"[sft]   lr={lr:g}: NO METRICS (arm did not start?)")
            continue
        recs = [json.loads(l) for l in open(mf) if l.strip()]
        vals = [(r["step"], r["val"]) for r in recs if r.get("val") is not None]
        if not vals:
            print(f"[sft]   lr={lr:g}: no val records yet")
            continue
        bs, bv = min(vals, key=lambda t: t[1])
        print(f"[sft]   lr={lr:g}: best_val={bv:.4f} @step {bs} "
              f"(last logged @step {vals[-1][0]})")
        if best is None or bv < best[1]:
            best = (lr, bv)
    if best:
        print(f"\n[sft] BEST_LR = {best[0]:g}  (best_val={best[1]:.4f})")
        print("[sft] paste it into BEST_LR above, then set MODE='final'")

## Final run — persistent ckpt_dir, resumable across sessions

Unlike the sweep arms this one is **not** wiped: re-running after a session kill
continues from `--resume latest` instead of starting over.


In [ ]:
import os, subprocess

if MODE != "final":
    print("skip: final runs in final mode only")
elif BEST_LR is None:
    print("BEST_LR is None: run MODE='sweep' and paste the winner into the flags cell.")
else:
    CKPT_DIR = "/kaggle/working/checkpoints/sft_final"
    os.makedirs(CKPT_DIR, exist_ok=True)
    log_run("final", CKPT_DIR, BEST_LR)
    argv = ["python", "-m", "train.sft",
            "--config", "configs/base_30m.yaml",
            "--init_weights", W, "--ckpt_dir", CKPT_DIR,
            "--max_examples", str(MAX_EXAMPLES), "--epochs", str(EPOCHS),
            "--batch_seqs", str(BATCH_SEQS), "--lr", str(BEST_LR),
            "--seed", str(SEED), "--save_every", "0",
            "--save_every_min", str(SAVE_EVERY_MIN),
            "--max_hours", str(MAX_HOURS), "--resume", "latest"]
    proc = subprocess.run(argv)
    if proc.returncode != 0:
        raise RuntimeError(f"[sft] FAILED: train.sft exited with code {proc.returncode}")

## Baseline comparison (final only)

Regenerates the same 5 prompts/settings as
`checkpoints/glassbox-s3-final/stories_256.txt` so SFT can be compared with the
pretraining baseline. Writes a **new** file and never overwrites the baseline.


In [ ]:
import glob, os, re, shutil, subprocess

if MODE != "final":
    print("skip: comparison runs in final mode only")
else:
    ws = sorted(glob.glob("/kaggle/working/checkpoints/sft_final/weights_step*.pt"),
                key=lambda x: int(re.search(r"weights_step(\d+)", x).group(1)))
    if not ws:
        print("no SFT weights yet")
    else:
        # Written to a file with the path passed as argv: no nested quoting.
        script = r"""import sys, torch
sys.path.insert(0, ".")
from common.config import load_config
from model.gpt import GPT
from tokenizer.bpe import BPETokenizer
from train.sample import sample_continuations

cfg = load_config("configs/base_30m.yaml")
tok = BPETokenizer.load(cfg["data"]["tokenizer_path"])
m = GPT(cfg["model"]).cpu()
m.load_state_dict(torch.load(sys.argv[1], map_location="cpu", weights_only=True))
m.eval()
# TEMPLATE header: the first 5 synthetic prompts of the SFT train stream
# (exact prompts seen during SFT: instruct.synth_prompt, stories 0-4).
PROMPTS_T = [
    ("Write a short story using all of these words: fixing, sharp, they.\n\nStory:", ("fixing", "sharp", "they")),
    ("Write a short story using all of these words: lived, played, when.\n\nStory:", ("lived", "played", "when")),
    ("Write a short story using all of these words: crab, fine, near.\n\nStory:", ("crab", "fine", "near")),
    ("Write a short story using all of these words: there, tickle, tree.\n\nStory:", ("there", "tickle", "tree")),
    ("Write a short story using all of these words: ever, little, were.\n\nStory:", ("ever", "little", "were")),
]
PROMPTS = ["Once upon a time,",
           "One day, Lily and her brother Tom went to the park.",
           "The little bear was lost in the big forest.",
           "Sara wanted very much to learn how to ride a bicycle.",
           "There was a small dog who lived on a farm. One morning,"]
with open("/tmp/sft_samples.txt", "w") as out:
    for i, p in enumerate(PROMPTS):
        txt = sample_continuations(m, tok, [p], max_new=256, temperature=0.8,
                                   top_k=40, seed=i, device="cpu")[0]
        out.write("\n##### prompt=%r seed=%d #####\n" % (p, i))
        out.write(txt.strip() + "\n")
    for i, (pt, wk) in enumerate(PROMPTS_T):
        txt = sample_continuations(m, tok, [pt], max_new=256, temperature=0.8,
                                   top_k=40, seed=100 + i, device="cpu")[0]
        out.write("\n##### TEMPLATE prompt=%r keywords=%r seed=%d #####\n"
                  % (pt, list(wk), 100 + i))
        out.write(txt.strip() + "\n")
print("wrote /tmp/sft_samples.txt")
"""
        open("/tmp/gen_sft_samples.py", "w").write(script)
        proc = subprocess.run(["python", "/tmp/gen_sft_samples.py", ws[-1]])
        if proc.returncode != 0:
            raise RuntimeError(f"[sft] FAILED: sampler exited with code {proc.returncode}")
        dst = "/kaggle/working/sft_samples_256.txt"  # ABSOLUTE, output root
        shutil.copy("/tmp/sft_samples.txt", dst)
        print("saved", dst, os.path.getsize(dst), "bytes")
        print("compare against checkpoints/glassbox-s3-final/stories_256.txt; never overwrite it")

## Backup (final only): newest SFT weights -> private HF repo, graceful skip

HF_TOKEN goes in Kaggle Secrets. Weights-only copies also ride along on Save Version.


In [ ]:
import glob, os

if MODE != "final":
    print("skip: backup runs in final mode only")
else:
    from kaggle_secrets import UserSecretsClient
    try:
        tok = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        tok = None
        print("no HF_TOKEN secret; skipping backup:", e)
    ckpts = sorted(glob.glob("/kaggle/working/checkpoints/sft_final/ckpt_step*.pt"))
    if tok and ckpts:
        try:
            import importlib
            try:
                hhub = importlib.import_module("huggingface_hub")
            except ImportError:
                !pip install -q huggingface_hub
                hhub = importlib.import_module("huggingface_hub")
            hhub.login(token=tok)
            api = hhub.HfApi()
            repo = api.create_repo("minigpt-stack-sft", private=True, exist_ok=True).repo_id
            for f in ckpts[-2:]:
                api.upload_file(path_or_fileobj=f, path_in_repo=os.path.basename(f), repo_id=repo)
            print("backed up to", repo, ckpts[-2:])
        except Exception as e:
            print("backup skipped (retry next session):", e)
    elif not ckpts:
        print("no checkpoints to back up")